# 02 · Working with Files & Folders: `os`, `shutil`, `pathlib`, `glob`

**Module 11 – Files & OS** · Estimated time: 55 min

> Merges 📗 *Opening and Reading Files and Folders* (os, shutil, send2trash, os.walk) with 📘 *OS and Shutil Modules*, and adds ✅ **`pathlib`** (the modern, recommended API, missing from both), `glob` and `tempfile`.

### 🎯 What you will learn
- `os`: current directory, listing, creating, renaming, removing, walking trees, environment variables
- `os.path`: joining, checking, splitting paths
- `shutil`: copy, move, delete trees, disk usage
- Safe deletion with `send2trash`
- ✅ `pathlib.Path`: object-oriented paths (🆕 3.14 `copy`/`move` methods)
- `glob` patterns and `tempfile`

## 1. The `os` module

In [1]:
import os

print("Current directory:", os.getcwd())
os.makedirs("practice/sub/deeper", exist_ok=True)        # nested folders; no error if they exist
os.mkdir("single_level") if not os.path.exists("single_level") else None
with open("practice/practice.txt", "w", encoding="utf-8") as f:
    f.write("This is a test string")
print(os.listdir("practice"))                             # files and folders in a directory
print(sorted(os.listdir(".")))

Current directory: /private/var/folders/96/f6xk04z91_gdystl3qfqv33r0000gn/T/tmp8syzicek
['practice.txt', 'sub']
['Find_the_Phone_Number.pdf', 'Working_Business_Proposal.pdf', 'blue_color.png', 'example.csv', 'example.jpg', 'find_the_link.csv', 'mask.png', 'pencils.jpg', 'practice', 'purple.png', 'red_color.jpg', 'single_level', 'word_matrix.png']


In [2]:
os.rename("practice/practice.txt", "practice/renamed.txt")
print(os.listdir("practice"))
os.remove("practice/renamed.txt")         # delete a FILE
os.rmdir("single_level")                  # delete an EMPTY folder
print(os.path.exists("single_level"))

['sub', 'renamed.txt']
False


### `os.path` helpers

In [3]:
path = os.path.join("folder", "subfolder", "file.txt")     # correct separator for the OS
print("Joined path:", path)
print(os.path.basename(path), "|", os.path.dirname(path))
print(os.path.splitext("report.final.pdf"))
print(os.path.exists("practice"), os.path.isdir("practice"), os.path.isfile("practice"))
print(os.path.abspath("."))
print(os.path.expanduser("~"))

Joined path: folder/subfolder/file.txt
file.txt | folder/subfolder
('report.final', '.pdf')
True True False
/private/var/folders/96/f6xk04z91_gdystl3qfqv33r0000gn/T/tmp8syzicek
/Users/gaurvendrasingh


### Environment variables

In [4]:
print(os.environ.get("HOME") or os.environ.get("USERPROFILE"))
print(os.environ.get("MY_API_KEY", "not set"))     # ✅ read secrets from env vars, never hard-code them
print(os.cpu_count(), os.sep, os.name)

/Users/gaurvendrasingh
not set
8 / posix


## 2. Walking through a directory tree: `os.walk` (📗)

In [5]:
for name in ["Example_Top_Level/Mid-Example-One/Bottom-Level-One",
             "Example_Top_Level/Mid-Example-One/Bottom-Level-Two"]:
    os.makedirs(name, exist_ok=True)
for file, text in {"Example_Top_Level/Mid-Example.txt": "mid",
                   "Example_Top_Level/Mid-Example-One/Mid-Level-Doc.txt": "doc",
                   "Example_Top_Level/Mid-Example-One/Bottom-Level-One/One_Text.txt": "one",
                   "Example_Top_Level/Mid-Example-One/Bottom-Level-Two/Bottom-Text-Two.txt": "two"}.items():
    with open(file, "w", encoding="utf-8") as f:
        f.write(text)

for folder, sub_folders, files in os.walk("Example_Top_Level"):
    print("Currently looking at folder:", folder)
    print("  Subfolders:", sub_folders)
    print("  Files:", files)

Currently looking at folder: Example_Top_Level
  Subfolders: ['Mid-Example-One']
  Files: ['Mid-Example.txt']
Currently looking at folder: Example_Top_Level/Mid-Example-One
  Subfolders: ['Bottom-Level-Two', 'Bottom-Level-One']
  Files: ['Mid-Level-Doc.txt']
Currently looking at folder: Example_Top_Level/Mid-Example-One/Bottom-Level-Two
  Subfolders: []
  Files: ['Bottom-Text-Two.txt']
Currently looking at folder: Example_Top_Level/Mid-Example-One/Bottom-Level-One
  Subfolders: []
  Files: ['One_Text.txt']


## 3. `shutil`: high-level operations

In [6]:
import shutil

with open("my_file.txt", "w", encoding="utf-8") as f:
    f.write("copy me")
shutil.copy("my_file.txt", "my_file_copy.txt")          # copy a file (copy2 also keeps metadata)
shutil.copytree("Example_Top_Level", "Backup", dirs_exist_ok=True)   # copy a whole folder
os.makedirs("new_directory", exist_ok=True)
shutil.move("my_file_copy.txt", "new_directory/")        # move (or rename)
print(os.listdir("new_directory"), os.path.exists("Backup/Mid-Example-One"))
shutil.rmtree("Backup")                                  # ⚠️ delete a folder AND everything inside
total, used, free = shutil.disk_usage(".")
print(f"Free disk: {free / 1e9:.1f} GB")
print(shutil.which("python3"))                           # find an executable on PATH

['my_file_copy.txt'] True
Free disk: 65.6 GB
/opt/homebrew/bin/python3


> ⚠️ `os.remove`, `os.rmdir`, `os.unlink`, `shutil.rmtree` delete **permanently**: no Recycle Bin, no undo.

## 4. Safer deletion: `send2trash` (📗)

`pip install send2trash` moves files to the **Recycle Bin / Trash** instead of deleting them permanently.

In [ ]:
import send2trash
with open("practice.txt", "w") as f:
    f.write("This is a test string")
send2trash.send2trash("practice.txt")      # now in your Trash, recoverable

## 5. ✅ `pathlib`: the modern way (3.4+, recommended)

`Path` objects represent paths and have methods for everything. They work identically on Windows, macOS and Linux.

In [7]:
from pathlib import Path

p = Path("reports") / "2026" / "summary.txt"     # the / operator joins paths!
print(p, type(p).__name__)
print("name:", p.name, "| stem:", p.stem, "| suffix:", p.suffix, "| parent:", p.parent)
print("parts:", p.parts)
print(p.with_suffix(".csv"), p.with_name("detail.txt"))
print(Path.cwd().name, "|", Path.home())

reports/2026/summary.txt PosixPath
name: summary.txt | stem: summary | suffix: .txt | parent: reports/2026
parts: ('reports', '2026', 'summary.txt')
reports/2026/summary.csv reports/2026/detail.txt
tmp8syzicek | /Users/gaurvendrasingh


In [8]:
p.parent.mkdir(parents=True, exist_ok=True)         # like os.makedirs
p.write_text("Revenue up 12%\nCosts down 3%\n", encoding="utf-8")   # open+write+close in one call
print(p.read_text(encoding="utf-8"))
print(p.exists(), p.is_file(), p.parent.is_dir(), p.stat().st_size, "bytes")
print(p.resolve())                                  # absolute path

Revenue up 12%
Costs down 3%

True True True 29 bytes
/private/var/folders/96/f6xk04z91_gdystl3qfqv33r0000gn/T/tmp8syzicek/reports/2026/summary.txt


In [9]:
# Listing & searching
for item in sorted(Path("Example_Top_Level").iterdir()):
    print("DIR " if item.is_dir() else "FILE", item.name)
print([f.name for f in Path("Example_Top_Level").rglob("*.txt")])   # recursive glob
print([f.name for f in Path(".").glob("*.txt")])

DIR  Mid-Example-One
FILE Mid-Example.txt
['Mid-Example.txt', 'Mid-Level-Doc.txt', 'Bottom-Text-Two.txt', 'One_Text.txt']
['my_file.txt']


In [10]:
# Path.walk() (3.12+): pathlib's version of os.walk
for folder, dirs, files in Path("Example_Top_Level").walk():
    print(folder, files)

Example_Top_Level ['Mid-Example.txt']
Example_Top_Level/Mid-Example-One ['Mid-Level-Doc.txt']
Example_Top_Level/Mid-Example-One/Bottom-Level-Two ['Bottom-Text-Two.txt']
Example_Top_Level/Mid-Example-One/Bottom-Level-One ['One_Text.txt']


### 🆕 Python 3.14: copy & move directly from `Path`

In [11]:
src = Path("reports/2026/summary.txt")
copied = src.copy(Path("reports/summary_copy.txt"))      # copy a file (or a whole directory tree)
moved = copied.move(Path("reports/archive.txt"))         # move / rename
print(moved, moved.exists(), copied.exists())
src.copy_into(Path("new_directory"))                     # copy INTO a folder, keeping the name
print(sorted(f.name for f in Path("new_directory").iterdir()))

reports/archive.txt True False
['my_file_copy.txt', 'summary.txt']


### Deleting with pathlib

In [12]:
Path("reports/archive.txt").unlink()                     # delete a file
Path("reports/ghost.txt").unlink(missing_ok=True)        # no error if missing
print(sorted(str(x) for x in Path("reports").rglob("*")))

['reports/2026', 'reports/2026/summary.txt']


### `os` / `os.path` ↔ `pathlib` cheat-sheet

| Task | `os` / `shutil` | `pathlib` |
|---|---|---|
| join | `os.path.join(a, b)` | `Path(a) / b` |
| cwd | `os.getcwd()` | `Path.cwd()` |
| exists | `os.path.exists(p)` | `p.exists()` |
| is file/dir | `os.path.isfile/isdir` | `p.is_file()` / `p.is_dir()` |
| make dirs | `os.makedirs(p, exist_ok=True)` | `p.mkdir(parents=True, exist_ok=True)` |
| list | `os.listdir(p)` | `p.iterdir()` |
| find files | `glob.glob("*.txt")` | `p.glob("*.txt")`, `p.rglob(...)` |
| read text | `open(p).read()` | `p.read_text()` |
| file name / ext | `os.path.basename`, `splitext` | `p.name`, `p.stem`, `p.suffix` |
| delete file | `os.remove(p)` | `p.unlink()` |
| rename | `os.rename(a, b)` | `p.rename(b)` |
| copy / move | `shutil.copy`, `shutil.move` | 🆕 `p.copy(dst)`, `p.move(dst)` (3.14) |

## 6. `glob`: wildcard patterns

| Pattern | Matches |
|---|---|
| `*` | any characters (not `/`) |
| `?` | exactly one character |
| `[abc]` | one of a, b, c |
| `**` | any number of folders (with `recursive=True`) |

In [13]:
import glob
print(glob.glob("*.txt"))
print(glob.glob("Example_Top_Level/**/*.txt", recursive=True))

['my_file.txt']
['Example_Top_Level/Mid-Example.txt', 'Example_Top_Level/Mid-Example-One/Mid-Level-Doc.txt', 'Example_Top_Level/Mid-Example-One/Bottom-Level-Two/Bottom-Text-Two.txt', 'Example_Top_Level/Mid-Example-One/Bottom-Level-One/One_Text.txt']


## 7. `tempfile`: temporary files & folders

In [14]:
import tempfile
with tempfile.TemporaryDirectory() as tmp:
    tmp_file = Path(tmp) / "scratch.txt"
    tmp_file.write_text("temporary!")
    print(tmp_file.exists(), tmp_file.read_text())
print("after with:", tmp_file.exists())               # cleaned up automatically

True temporary!
after with: False


## 8. Practical example: organise a messy folder by file type

In [15]:
downloads = Path("Downloads_demo")
downloads.mkdir(exist_ok=True)
for name in ["photo.jpg", "cv.pdf", "song.mp3", "notes.txt", "scan.PDF", "pic.png"]:
    (downloads / name).write_text("x")

categories = {"Images": {".jpg", ".png"}, "Documents": {".pdf", ".txt"}, "Music": {".mp3"}}
for file in downloads.iterdir():
    if file.is_file():
        for folder, exts in categories.items():
            if file.suffix.lower() in exts:
                target = downloads / folder
                target.mkdir(exist_ok=True)
                file.rename(target / file.name)
                break

for folder in sorted(downloads.iterdir()):
    print(folder.name, sorted(f.name for f in folder.iterdir()))

Documents ['cv.pdf', 'notes.txt', 'scan.PDF']
Images ['photo.jpg', 'pic.png']
Music ['song.mp3']


## 📝 Summary
- `os`: cwd, listdir, makedirs, rename, remove, walk, environ.
- `shutil`: copy, copytree, move, rmtree (⚠️ permanent), disk_usage.
- `send2trash` for recoverable deletes.
- ✅ Prefer `pathlib.Path` in new code: `/` joining, `read_text`, `glob`, 🆕 `copy`/`move`.

## 🏋️ Practice
1. Print the 5 largest files (name and size) anywhere under a folder of your choice.
2. Rename every `.txt` file in a folder to have a `2026_` prefix.
3. Count files per extension under a folder (`{'.py': 10, '.txt': 3, ...}`).
4. Delete all empty sub-folders of a folder.

<details><summary>Show solutions</summary>

```python
from pathlib import Path
from collections import Counter

root = Path(".")
files = [f for f in root.rglob("*") if f.is_file()]
for f in sorted(files, key=lambda f: f.stat().st_size, reverse=True)[:5]:
    print(f, f.stat().st_size)

for f in Path("some_folder").glob("*.txt"):
    f.rename(f.with_name("2026_" + f.name))

print(Counter(f.suffix.lower() for f in files))

for d in sorted((d for d in root.rglob("*") if d.is_dir()), key=lambda d: len(d.parts), reverse=True):
    if not any(d.iterdir()):
        d.rmdir()
```
</details>